# 01b · Confronto tra modelli

Nel notebook precedente un albero di decisione è arrivato al 99,6% di accuratezza. Qui ci chiediamo:
- **altri modelli fanno meglio o peggio?** E a che prezzo, in tempo di addestramento e di risposta?
- **si può riconoscere un attacco senza averne mai visto uno?** È l'idea dei modelli non supervisionati, che imparano solo com'è fatto il traffico normale;
- **come si scelgono gli iperparametri** di un modello senza imbrogliare?
- **un solo esperimento basta?**

**Come usare il notebook.** Esegui le celle in ordine con `Shift + Invio`. I riquadri **Prova tu** sono piccole modifiche da fare in aula. Le **Sfide** in fondo sono da fare a casa.

## 0. Setup

In [ ]:
# Cella di setup: eseguila per prima, ogni volta che apri il notebook.
import subprocess, sys
from pathlib import Path

REPO = "salvatore-drago/nids-ml-lab"

if "google.colab" in sys.modules:
    root = Path("/content/nids-ml-lab")
    if not root.exists():
        url = f"https://github.com/{REPO}.git"
        try:  # repository privato: token nei Secrets di Colab, con nome GITHUB_TOKEN
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            pass
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(root)]).returncode:
            raise RuntimeError("Clone del repository non riuscito: controlla la connessione o l'accesso.")
    else:
        subprocess.run(["git", "-C", str(root), "pull", "-q"])
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nidslab").is_dir()), None)
    if root is None:
        raise RuntimeError("Apri il notebook dalla cartella notebooks/ del repository nids-ml-lab.")

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import nidslab as nl

nl.ensure_packages()
nl.set_style()
print("Repository:", root)

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import ParameterSampler, train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42

## 1. I dati

Ripartiamo dalle decisioni del notebook `01a`: niente identificativi, niente colonne costanti, niente duplicati, flussi di un pacchetto tenuti. La funzione `nl.prepare_static()` fa esattamente quei passaggi.

Questa volta dividiamo i dati in **tre** parti, stratificate sul tipo di attacco:
- **addestramento** (20%): il modello impara da qui;
- **validazione** (10%): serve a confrontare modelli e iperparametri, cioè a prendere decisioni;
- **test** (70%): lo usiamo per misurare il risultato finale, e mai per prendere decisioni.

Perché non basta il test? Se scegliamo il modello migliore guardando il test, il test ha partecipato alla scelta, e il suo risultato non è più una stima onesta di come andrà il modello su dati nuovi.

In [ ]:
data, features = nl.prepare_static()
X, y, lab = data[features], data["label_binary"], data["label"]


def split(seed):
    """Divide i dati in addestramento (20%), validazione (10%) e test (70%)."""
    X_tr, X_rest, y_tr, y_rest, l_tr, l_rest = train_test_split(
        X, y, lab, train_size=0.20, stratify=lab, random_state=seed)
    X_va, X_te, y_va, y_te, l_va, l_te = train_test_split(
        X_rest, y_rest, l_rest, train_size=0.125, stratify=l_rest, random_state=seed)
    return X_tr, X_va, X_te, y_tr, y_va, y_te, l_tr, l_va, l_te


X_train, X_val, X_test, y_train, y_val, y_test, lab_train, lab_val, lab_test = split(SEED)
print(f"{len(features)} feature")
print(f"Addestramento: {len(X_train):>7,} flussi, attacchi {y_train.mean():.1%}")
print(f"Validazione:   {len(X_val):>7,} flussi, attacchi {y_val.mean():.1%}")
print(f"Test:          {len(X_test):>7,} flussi, attacchi {y_test.mean():.1%}")

> **Da notare.** Grazie alla stratificazione, le tre parti hanno la stessa quota di attacchi, il 10,2%. L'addestramento contiene esattamente gli stessi 47.264 flussi del notebook `01a`.

Per confrontare i modelli useremo sempre le stesse misure:
- **accuratezza**: quota di flussi classificati correttamente;
- **attacchi riconosciuti**: quota di flussi d'attacco segnalati come attacco (detto anche *recall* o *detection rate*);
- **falsi allarmi**: quota di flussi benigni segnalati come attacco;
- **F1 attacco**: una media tra la quota di attacchi riconosciuti e la quota di allarmi che sono attacchi veri (la *precision*). La useremo per scegliere tra modelli; in `01c` vedremo perché e quali alternative esistono;
- **tempi** di addestramento e di predizione.

In [ ]:
def scores(y_true, pred):
    y_true, pred = np.asarray(y_true), np.asarray(pred)
    return {
        "accuratezza": accuracy_score(y_true, pred),
        "attacchi riconosciuti": pred[y_true == 1].mean(),
        "falsi allarmi": pred[y_true == 0].mean(),
        "F1 attacco": f1_score(y_true, pred),
    }


def evaluate(name, model, X_tr, y_tr, X_ev, y_ev):
    """Addestra il modello, lo valuta e restituisce una riga di risultati e le predizioni."""
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    t_fit = time.perf_counter() - t0
    t0 = time.perf_counter()
    pred = np.asarray(model.predict(X_ev)).astype(int)
    t_pred = time.perf_counter() - t0
    row = {"modello": name, **scores(y_ev, pred), "addestramento (s)": t_fit, "predizione (s)": t_pred}
    return row, pred


def show(rows):
    table = pd.DataFrame(rows).set_index("modello")
    return table.round({"accuratezza": 4, "attacchi riconosciuti": 4, "falsi allarmi": 4,
                        "F1 attacco": 4, "addestramento (s)": 1, "predizione (s)": 2})

## 2. Modelli supervisionati

Cinque modelli che imparano da esempi etichettati, cioè sapendo per ogni flusso se è un attacco:
- **albero di decisione**: domande in sequenza, come in `01a`;
- **random forest**: cento alberi diversi che votano;
- **gradient boosting**: alberi piccoli in sequenza, ciascuno corregge gli errori dei precedenti;
- **regressione logistica**: una somma pesata delle feature, trasformata in probabilità. Traccia una frontiera dritta;
- **rete neurale (MLP)**: strati di unità che combinano le feature, come nelle slide.

Gli alberi ragionano per soglie e non si accorgono della scala delle feature. Regressione logistica e rete neurale invece fanno somme pesate: una feature che vale milioni (i byte) schiaccia una che vale 0 o 1 (un flag). Per loro **standardizziamo**: prima comprimiamo le code con $\log(1 + x)$, poi portiamo ogni feature a media 0 e deviazione standard 1.

> **Prova tu.** Metti `SCALE = False` ed esegui di nuovo la cella. Quali modelli cambiano, e quali no?

In [ ]:
SCALE = True


def log1p_positive(X):
    return np.log1p(np.clip(X, 0, None))


def scaled(model):
    """Aggiunge la standardizzazione davanti al modello, se SCALE è True."""
    if not SCALE:
        return model
    return make_pipeline(FunctionTransformer(log1p_positive), StandardScaler(), model)


supervised = {
    "Albero di decisione": DecisionTreeClassifier(random_state=SEED),
    "Random forest": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
    "Gradient boosting": HistGradientBoostingClassifier(random_state=SEED),
    "Regressione logistica": scaled(LogisticRegression(max_iter=1000)),
    "Rete neurale (MLP)": scaled(MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True,
                                               max_iter=200, random_state=SEED)),
}

rows, val_pred = [], {}
for name, model in supervised.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        row, val_pred[name] = evaluate(name, model, X_train, y_train, X_val, y_val)
    if any("converge" in str(w.message).lower() for w in caught):
        print(f"Attenzione: {name} non ha raggiunto la convergenza")
    rows.append(row)
    print(f"{name:<24} fatto in {row['addestramento (s)']:.1f} s")

supervised_results = rows
show(rows)

Questa tabella è calcolata sul set di **validazione**: la usiamo per confrontare i modelli e scegliere.

> **Da notare** (con `SCALE = True`).
> - Il **gradient boosting** vince su tutto: F1 0,989, 98,6% di attacchi riconosciuti e lo 0,08% di falsi allarmi, **un terzo di quelli dell'albero**. Ed è anche il più veloce da addestrare tra i modelli ad albero.
> - La **random forest** riconosce meno attacchi dell'albero singolo (97,7% contro 98,2%) ma fa meno falsi allarmi (0,17% contro 0,25%): cento alberi che votano sono più prudenti di uno solo, perché per segnalare un attacco serve la maggioranza.
> - La random forest si addestra **più in fretta** dell'albero singolo. Lavora in parallelo su tutti i processori, e a ogni divisione ogni albero considera solo una decina di feature scelte a caso (circa $\sqrt{111}$), mentre l'albero singolo le esamina tutte.
> - La **regressione logistica** è chiaramente indietro: 92% di attacchi riconosciuti e falsi allarmi quasi doppi rispetto all'albero. Una frontiera dritta non basta a separare questo traffico.
> - I tempi di **predizione** sono trascurabili per tutti: meno di 0,15 secondi per oltre 23 mila flussi.
>
> **Dopo il Prova tu** (con `SCALE = False`).
> - Albero, random forest e gradient boosting hanno **metriche identiche** fino all'ultima cifra: ragionano per soglie, e cambiare la scala di una feature sposta solo la soglia.
> - La regressione logistica crolla: gli attacchi riconosciuti scendono dal 92% al **75%**, l'ottimizzazione non converge e l'addestramento passa da 3 a oltre 16 secondi.
> - Anche la rete neurale peggiora molto: dal 97% all'85% di attacchi riconosciuti, e i falsi allarmi quasi quadruplicano. L'addestramento è più breve solo perché l'*early stopping* si arrende prima.
> - I tempi misurati cambiano da un'esecuzione all'altra, anche con metriche identiche: vanno letti come ordini di grandezza.

## 3. Modelli non supervisionati: imparare il normale

I modelli della sezione 2 riconoscono gli attacchi **che hanno visto**. Un attacco nuovo, mai entrato nei dati di addestramento, potrebbe sfuggire. L'idea dell'*anomaly detection* è rovesciare il problema: il modello impara **solo il traffico benigno**, e segnala tutto ciò che se ne discosta troppo.
- **Isolation Forest** isola ogni punto con tagli casuali: i punti anomali, lontani dagli altri, si isolano con pochi tagli;
- **Local Outlier Factor (LOF)** confronta la densità attorno a ogni punto con quella dei suoi vicini: un punto in una zona molto più vuota dei vicini è anomalo. Ragiona con le distanze, quindi va standardizzato, ed è il più lento dei due a rispondere, perché per ogni flusso deve cercare i vicini tra tutti quelli di addestramento.

Nessuno dei due usa le etichette degli attacchi. Le usiamo solo per misurare come vanno.

La soglia oltre la quale un punto è "anomalo" si regola con `contamination`: è la quota di flussi **benigni** di addestramento che il modello considera anomali. In pratica è la quota di falsi allarmi che accettiamo.

> **Prova tu.** Cambia `CONTAMINATION` (0.005, 0.02, 0.05, 0.10) e riesegui. Come cambiano insieme attacchi riconosciuti e falsi allarmi?

In [ ]:
CONTAMINATION = 0.02

benign_train = X_train[y_train == 0]


class AnomalyDetector:
    """Adatta un detector di anomalie (che risponde -1 = anomalo, 1 = normale)
    all'interfaccia dei classificatori: 1 = attacco, 0 = benigno."""

    def __init__(self, detector, fit_data):
        self.detector, self.fit_data = detector, fit_data

    def fit(self, X, y=None):
        self.detector.fit(self.fit_data)  # solo traffico benigno, nessuna etichetta
        return self

    def predict(self, X):
        return (self.detector.predict(X) == -1).astype(int)


unsupervised = {
    "Isolation Forest": AnomalyDetector(
        IsolationForest(n_estimators=200, contamination=CONTAMINATION, n_jobs=-1, random_state=SEED),
        benign_train),
    "LOF": AnomalyDetector(
        make_pipeline(FunctionTransformer(log1p_positive), StandardScaler(),
                      LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=CONTAMINATION, n_jobs=-1)),
        benign_train),
}

rows = []
for name, model in unsupervised.items():
    row, val_pred[name] = evaluate(name, model, X_train, y_train, X_val, y_val)
    rows.append(row)
    print(f"{name:<24} fatto in {row['addestramento (s)'] + row['predizione (s)']:.1f} s")

show(rows)

> **Da notare** (con `CONTAMINATION = 0.02`). I due modelli non trovano quasi niente: Isolation Forest segnala il 4% degli attacchi, LOF il 12%, con circa il 2% di falsi allarmi. In questo spazio di feature, il 2% di flussi benigni più insoliti (servizi rari, connessioni lunghissime) è più "anomalo" della maggior parte degli attacchi.
>
> Due prove fatte per rendere il confronto onesto:
> - Isolation Forest sui dati trasformati con $\log(1 + x)$ va peggio (2,7% di attacchi riconosciuti), quindi resta sui dati grezzi;
> - LOF addestrato su 10.000 benigni invece che su tutti scende allo 0,7%: per stimare la densità del "normale" servono molti dati.

Quali attacchi sono "anomali"? Confrontiamo, tipo per tipo, il miglior modello supervisionato (secondo l'F1 sulla validazione) con i due non supervisionati.

In [ ]:
best_supervised = max(supervised_results, key=lambda r: r["F1 attacco"])["modello"]
compare = [best_supervised, "Isolation Forest", "LOF"]

per_class = pd.DataFrame({name: val_pred[name] for name in compare})
per_class["label"] = lab_val.to_numpy()
table = per_class.groupby("label", observed=True)[compare].mean()
table.insert(0, "flussi", per_class.groupby("label", observed=True).size())
print(f"Miglior supervisionato: {best_supervised}")
print("Quota di flussi segnalati come attacco (per Benign: falsi allarmi)")
table.sort_values("flussi", ascending=False).round(3)

> **Da notare** (con `CONTAMINATION = 0.02`).
> - LOF trova circa metà dei **DoS lenti** (Slowhttptest 55%, Slowloris 48%). Sono attacchi che tengono aperte molte connessioni mandando richieste parziali, goccia a goccia: ogni singolo flusso ha durate e intervalli tra pacchetti davvero insoliti.
> - I flood e gli scan quasi no: DoS Hulk 13%, DDoS 9%, Port Scan 2%.
> - Il gradient boosting resta molto sopra in tutte le classi; le più difficili per lui sono FTP-Patator (81%) e Botnet (86%).
> - Nella validazione le classi più rare hanno pochissimi flussi (1 Heartbleed, 2 SQL Injection, 7 XSS): quelle righe non si possono leggere.
>
> **Dopo il Prova tu.**
>
> | `CONTAMINATION` | IF attacchi | IF falsi allarmi | LOF attacchi | LOF falsi allarmi |
> |---|---|---|---|---|
> | 0.005 | 0% | 0,6% | 0,8% | 0,4% |
> | 0.02 | 4,2% | 2,0% | 11,9% | 1,8% |
> | 0.05 | 75,9% | 5,0% | 57,6% | 4,8% |
> | 0.10 | 80,1% | 10,2% | 81,2% | 9,9% |
>
> - Tra il 2% e il 5% c'è un **salto**: entrano in massa flood e DDoS (con 0.05 Isolation Forest trova il 99,9% dei DDoS e l'80% dei DoS Hulk). Questi attacchi sono quindi **moderatamente** anomali: più insoliti del 95% del traffico benigno, ma meno della sua coda estrema.
> - I DoS lenti sono anomali in modo estremo: con 0.005 LOF trova ancora metà degli Slowloris.
> - Oltre il 5% i guadagni calano, e anche al 10% i web attack restano quasi invisibili e SSH-Patator non supera il 24%: assomigliano troppo al traffico web e SSH legittimo.
> - Il prezzo: per riconoscere l'80% degli attacchi senza averne mai visto uno, serve accettare **un falso allarme ogni 10 flussi benigni**. Il gradient boosting riconosce il 98,6% degli attacchi con un falso allarme ogni 1.250.

Un modello non supervisionato non sa che esistono i DoS o i port scan: segnala ciò che è **diverso dal normale**. Alcuni attacchi lo sono in modo evidente, altri somigliano molto al traffico benigno. In cambio, non ha bisogno di esempi d'attacco etichettati, e in linea di principio può accorgersi anche di un attacco che nessuno ha mai visto. Ma *anomalo* non vuol dire *malevolo*, e la scelta della soglia decide tutto. Torneremo su questo compromesso in `01c`.

## 4. Scegliere gli iperparametri

Ogni modello ha delle manopole che non impara da solo ma che dobbiamo fissare noi: la profondità di un albero, il numero di alberi di una foresta, la dimensione di una rete. Sono gli **iperparametri**. Si scelgono guardando la **validazione**, mai il test.

### 4.1 Una manopola alla volta: la profondità dell'albero

In [ ]:
depths = [1, 2, 3, 4, 6, 8, 10, 15, 20, 30, None]
curve = []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(X_train, y_train)
    curve.append({
        "max_depth": "∞" if d is None else str(d),
        "addestramento": f1_score(y_train, m.predict(X_train)),
        "validazione": f1_score(y_val, m.predict(X_val)),
    })
curve = pd.DataFrame(curve)

fig, ax = plt.subplots(figsize=(7, 4))
pos = np.arange(len(curve))
ax.plot(pos, curve["addestramento"], marker="o", label="Addestramento")
ax.plot(pos, curve["validazione"], marker="o", label="Validazione")
ax.set_xticks(pos, curve["max_depth"])
ax.set_xlabel("Profondità massima dell'albero")
ax.set_ylabel("F1 attacco")
ax.set_title("Curva di validazione dell'albero di decisione", loc="left")
ax.legend()
plt.show()

best_depth = curve.loc[curve["validazione"].idxmax(), "max_depth"]
print(f"Profondità migliore sulla validazione: {best_depth}")

È lo stesso fenomeno della sezione 3 di `01a`, ora sul dataset completo: oltre una certa profondità, l'albero continua a migliorare sui dati che ha visto ma non su quelli nuovi.

> **Da notare.**
> - Fino a profondità 10 le due curve salgono insieme. Da 15 in poi la validazione si ferma intorno a 0,98, mentre l'addestramento continua fino a quasi 1: la distanza tra le due curve è la parte che l'albero sta memorizzando.
> - La profondità migliore è 15, ma tra 15, 30 e senza limiti la validazione cambia di pochi decimillesimi, meno della variabilità che vedremo nella sezione 5. In pratica "da 15 in su è uguale", e a parità di risultato conviene il modello più semplice.
> - Già con **una sola domanda** l'albero arriva a un F1 di 0,85: un altro segno di quanto il compito sia facile.

### 4.2 Più manopole insieme: ricerca casuale

La random forest ha più iperparametri. Provare tutte le combinazioni (*grid search*) costa molto; spesso funziona altrettanto bene provarne alcune **a caso** (*random search*). Ogni combinazione viene addestrata sull'addestramento e valutata sulla validazione.

In [ ]:
N_TRIALS = 6

space = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.3],
}
trials = []
t0 = time.perf_counter()
for params in ParameterSampler(space, n_iter=N_TRIALS, random_state=SEED):
    m = RandomForestClassifier(**params, n_jobs=-1, random_state=SEED).fit(X_train, y_train)
    trials.append({**params, "F1 validazione": f1_score(y_val, m.predict(X_val))})
print(f"{N_TRIALS} combinazioni provate in {time.perf_counter() - t0:.0f} s")

trials = pd.DataFrame(trials).sort_values("F1 validazione", ascending=False)
trials

> **Da notare.**
> - La combinazione migliore (100 alberi, `max_features` 0,3, profondità 20) arriva a 0,986 sulla validazione, mezzo punto sopra la random forest di default della sezione 2.
> - Il parametro che conta di più è `max_features`: le prime due combinazioni usano entrambe 0,3, cioè circa 33 feature a ogni divisione invece di 10. Le peggiori sono quelle con profondità 10. Pochi iperparametri fanno la differenza, ed è proprio l'idea alla base della ricerca casuale.
> - Anche la combinazione migliore resta sotto il **gradient boosting senza nessuna ottimizzazione** (0,989): scegliere la famiglia di modelli giusta ha contato più che regolare quella sbagliata.

### 4.3 Il test, una volta sola

Ora che abbiamo scelto, e solo ora, misuriamo sul test il modello di default e quello con gli iperparametri migliori. Il test non ha partecipato a nessuna decisione, quindi questo è il numero da riportare.

In [ ]:
best_params = trials.iloc[0].drop("F1 validazione").to_dict()
best_params = {k: (None if pd.isna(v) else v) for k, v in best_params.items()}
for k in ("n_estimators", "max_depth", "min_samples_leaf"):
    if best_params[k] is not None:
        best_params[k] = int(best_params[k])
print("Iperparametri scelti:", best_params)

final = []
for name, model in {
    "Random forest, default": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
    "Random forest, scelta sulla validazione": RandomForestClassifier(**best_params, n_jobs=-1, random_state=SEED),
}.items():
    row, _ = evaluate(name, model, X_train, y_train, X_test, y_test)
    final.append(row)
show(final)

> **Da notare.**
> - La foresta scelta sulla validazione migliora su tutte le metriche, e soprattutto fa **un terzo di falsi allarmi in meno** (0,09% contro 0,13%).
> - Il guadagno di F1 sul test (+0,002) è meno della metà di quello visto sulla validazione (+0,005). È normale: tra sei combinazioni abbiamo scelto quella andata meglio sulla validazione, anche un po' per fortuna. Il test non ha partecipato alla scelta, e quella fortuna sparisce. Per questo il numero da riportare è quello del test.
> - Il prezzo è il tempo: l'addestramento è più di tre volte più lungo, perché a ogni divisione ogni albero esamina 33 feature invece di 10.

## 5. Un seed non basta

Tutti i numeri visti finora vengono da **una** divisione casuale dei dati, fissata da `SEED`. Con un'altra divisione sarebbero diversi. Di quanto? Ripetiamo l'esperimento con cinque divisioni diverse e guardiamo media e deviazione standard.

> **Prova tu.** Aumenta il numero di ripetizioni (`SEEDS`) o aggiungi un modello al dizionario. Le differenze tra i modelli sono più grandi o più piccole della variabilità tra un seed e l'altro?

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
models_to_repeat = {
    "Albero di decisione": lambda s: DecisionTreeClassifier(random_state=s),
    "Gradient boosting": lambda s: HistGradientBoostingClassifier(random_state=s),
}

runs = []
t0 = time.perf_counter()
for s in SEEDS:
    X_tr, X_va, X_te, y_tr, y_va, y_te, *_ = split(s)
    for name, make in models_to_repeat.items():
        row, _ = evaluate(name, make(s), X_tr, y_tr, X_te, y_te)
        runs.append({**row, "seed": s})
print(f"{len(runs)} addestramenti in {time.perf_counter() - t0:.0f} s")

runs = pd.DataFrame(runs)
metrics = ["accuratezza", "attacchi riconosciuti", "falsi allarmi", "F1 attacco"]
summary = runs.groupby("modello")[metrics].agg(["mean", "std"])
summary.round(4)

> **Da notare.**
> - Il gradient boosting è **più stabile**, oltre che più accurato: F1 0,9895 ± 0,0006 contro 0,9810 ± 0,0023 dell'albero. La variabilità dell'albero è quattro volte più grande: un singolo albero dipende molto da quali flussi gli capitano, un insieme di alberi media questi effetti.
> - La differenza tra i due modelli (0,0085) è molto più grande delle deviazioni standard: è reale, non dipende dal seed.
> - I due millesimi guadagnati dalla foresta ottimizzata nella sezione 4.3 sono invece dello **stesso ordine della variabilità** dell'albero: il miglioramento è plausibile, ma una sola divisione dei dati non basta per esserne sicuri. Questa tabella dà l'ordine di grandezza; per una risposta precisa bisognerebbe ripetere anche le due foreste su più seed (prova a casa, aggiungendole al dizionario).
> - Anche i falsi allarmi oscillano: 0,23% ± 0,04% per l'albero. Con un altro seed, i 345 falsi allarmi del notebook `01a` avrebbero potuto essere 290 o 400.

Un risultato va riportato con la sua variabilità: "0,99 ± 0,001" dice molto più di "0,99". E se due modelli differiscono meno della deviazione standard, non possiamo dire quale sia il migliore.

---

## Sfide (a casa)

Ogni sfida ha una cella con dei `TODO` da completare e un controllo (`assert`) che ti dice se hai impostato le cose nel modo giusto.

### Sfida 1: il k-nearest neighbors

Il *k-nearest neighbors* (kNN) classifica un flusso guardando i $k$ flussi di addestramento più vicini. Aggiungilo al confronto della sezione 2 con `KNeighborsClassifier(n_neighbors=5)`, standardizzato come la regressione logistica. Attenzione: la predizione è molto lenta, quindi valutalo su un campione di 20.000 flussi della validazione. Perché è così lento a predire, mentre l'addestramento è istantaneo?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

idx = X_val.sample(20_000, random_state=SEED).index if len(X_val) > 20_000 else X_val.index

# TODO: costruisci il modello kNN standardizzato e valutalo con evaluate() su X_val.loc[idx], y_val.loc[idx]
knn_row = ...

assert isinstance(knn_row, dict) and "F1 attacco" in knn_row, "knn_row deve essere la riga restituita da evaluate()"
show([knn_row])

### Sfida 2: un detector di anomalie fatto a mano

Un'altra idea di anomaly detection: una rete neurale che impara a **ricostruire** il traffico benigno, cioè a restituire in uscita lo stesso flusso che riceve in ingresso, passando per uno strato stretto. Su un flusso benigno la ricostruzione sarà buona; su un flusso diverso da tutti quelli visti, peggiore. L'**errore di ricostruzione** diventa il punteggio di anomalia. È l'idea degli *autoencoder*.

1. Standardizza i dati benigni di addestramento (come per LOF) e addestra `MLPRegressor(hidden_layer_sizes=(32, 8, 32))` con gli stessi dati come ingresso e come uscita.
2. Calcola l'errore quadratico medio di ricostruzione per ogni flusso della validazione.
3. Fissa la soglia al 98° percentile degli errori sui benigni di addestramento (cioè `contamination = 0.02`).
4. Confronta attacchi riconosciuti e falsi allarmi con Isolation Forest e LOF.

In [ ]:
from sklearn.neural_network import MLPRegressor

# TODO: errore di ricostruzione per ogni flusso della validazione (un numero per riga)
val_errors = ...

# TODO: la soglia, cioè il 98° percentile degli errori sui benigni di addestramento
threshold = ...

assert len(val_errors) == len(X_val), "Serve un errore di ricostruzione per ogni flusso della validazione"
ae_pred = (np.asarray(val_errors) > threshold).astype(int)
print(scores(y_val, ae_pred))

### Sfida 3: la cross-validation

Nella sezione 4.2 ogni combinazione è stata valutata su un solo set di validazione. In alternativa, la **cross-validation** divide l'addestramento in 5 parti e, a turno, addestra su quattro e valuta sulla quinta, facendo la media. Rifai la ricerca casuale con `RandomizedSearchCV` (stesso spazio, stesso `N_TRIALS`, `cv=5`, `scoring="f1"`). Gli iperparametri scelti cambiano? Quanto tempo in più serve? (Su Colab conta qualche minuto: sono 30 foreste da addestrare.)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# TODO: ricerca casuale con cross-validation a 5 fold sui dati di addestramento
search = ...

assert hasattr(search, "best_params_"), "search deve essere un RandomizedSearchCV già addestrato"
print("Iperparametri scelti con la cross-validation:", search.best_params_)
print("Iperparametri scelti con il set di validazione:", best_params)

### Sfida 4: la pulizia giusta dipende dal modello?

In `01a` abbiamo tolto i duplicati: per un classificatore sono ridondanza, e la stessa riga non deve stare sia nell'addestramento sia nella valutazione. Per un detector di anomalie, però, la **frequenza** è informazione: un flusso benigno che si ripete mille volte è il cuore del "normale". Togliendo le copie, il traffico benigno più tipico pesa quanto un flusso visto una volta sola, e il "normale" diventa meno denso.

Verifica l'ipotesi: riaddestra Isolation Forest e LOF sui benigni di addestramento **con le loro copie**, con la stessa `CONTAMINATION` della sezione 3, e confronta attacchi riconosciuti e falsi allarmi sulla validazione.

Attenzione a come rimetti i duplicati. Non vanno presi a caso dal dataset grezzo: alcune copie corrispondono a flussi che stanno nella validazione, e il detector li vedrebbe in anticipo. Rimetti **solo le copie dei flussi benigni che sono già nell'addestramento**, riconoscendole dall'impronta (hash) delle feature, come nella sezione 2.4 di `01a`.

Una domanda in più: LOF stima la densità attorno a un punto guardando i suoi 20 vicini più prossimi. Cosa succede a quella stima per un flusso che, con le copie, compare più di 20 volte?

In [ ]:
raw = nl.load_static()
raw_fp = pd.util.hash_pandas_object(raw[features], index=False)
train_benign_fp = set(pd.util.hash_pandas_object(benign_train, index=False))

# TODO: le righe benigne di `raw` la cui impronta è in train_benign_fp, con tutte le loro copie (solo le colonne `features`)
benign_train_dup = ...

assert len(benign_train_dup) >= len(benign_train), "Con le copie le righe devono essere almeno quante prima"
assert list(benign_train_dup.columns) == features, "Usa solo le colonne in `features`"

# TODO: addestra Isolation Forest e LOF su benign_train_dup con la stessa CONTAMINATION della sezione 3
# e confronta attacchi riconosciuti e falsi allarmi sulla validazione con quelli della sezione 3